**1. Import different config files**
We will start the notebook with OmegaConfig, which exists to combine multiple configuration files into one tree. This optimizes workflow and reduces the need for a crowded hierarchy of config files. Features include reading the tree with dotlist notation, merging with overrides, and checking variables against a typed schema.

In [ ]:
from pathlib import Path
from omegaconf import OmegaConf

base_dir = Path.cwd()

market = OmegaConf.load(str(base_dir / "conf" / "market" / "aapl.yaml"))
strategy = OmegaConf.load(str(base_dir / "conf" / "strategy" / "sma_crossover.yaml"))
backtest = OmegaConf.load(str(base_dir / "conf" / "backtest" / "default.yaml"))

# tree made from market.yaml, strategy.yaml, backtest.yaml
cfg = OmegaConf.create({"market": market, "strategy": strategy, "backtest": backtest})
print(OmegaConf.to_yaml(cfg))
print(cfg.market.ticker, cfg.strategy.fast_window)

: 

In [ ]:
dotlist = ["strategy.fast_window=5", "backtest.commission=0.002"]
override = OmegaConf.from_dotlist(dotlist)
cfg = OmegaConf.merge(cfg, override)
print(cfg.strategy.fast_window, cfg.backtest.commission)   # 5 0.002

In [ ]:
@dataclass
class StrategyConfig:
    name: str
    fast_window: int
    slow_window: int

schema = OmegaConf.structured(StrategyConfig)
checked = OmegaConf.merge(schema, strategy)                # passes
print(checked.slow_window)                                 # 30

# Break it on purpose, one line at a time (each raises, so run them separately):
OmegaConf.merge(schema, {"fast_window": "quick"})          # ValidationError: not an int
OmegaConf.merge(schema, {"fast_windw": 5})                 # ConfigKeyError: typo caught

In [ ]:
'''
python run.py strategy.fast_window=5 backtest.commission=0.002   # change values
python run.py market=tsla                                        # swap a whole group option
python run.py +backtest.note=trial1                              # + adds a key that does not exist
python run.py strategy.fast_window=50                            # trips the fast < slow check
python run.py --cfg job                                          # print the config, do not run main()
'''

In [ ]:
# python run.py -m market=aapl,msft,tsla
# python run.py -m strategy.fast_window=5,10,20
# python run.py -m market=aapl,tsla strategy.fast_window=5,20